# Download Daymet

Daymet provides gridded meteorological data for North American at 1km spatial resolution with daily timestep from 1980 ~ present. [website](https://daac.ornl.gov/cgi-bin/dsviewer.pl?ds_id=1328) and [user guide](https://daac.ornl.gov/DAYMET/guides/Daymet_V3_CFMosaics.html)

Available variables:

| Variable | Description (units) |
| ---- | ---- |
| tmax | Daily maximum 2-meter air temperature (°C) |
| tmin | Daily minimum 2-meter air temperature (°C) |
| prcp | Daily total precipitation (mm/day) |
| srad | Incident shortwave radiation flux density (W/m2) |
| vp   | Water vapor pressure (Pa) |
| swe  | Snow water equivalent (kg/m2) |
| dayl | Duration of the daylight period (seconds/day) |

Notes:
 - The Daymet calendar is based on a standard calendar year. All Daymet years, including leap years, have 1 - 365 days. For leap years, the Daymet database includes leap day (February 29) and values for December 31 are discarded from leap years to maintain a 365-day year.

In [1]:
# Use branch v1.3. use env watershed_workflow_daymet
# use the watershed_workflowrc to set the chrdir to the data directory

import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 150
import subprocess
import os

In [2]:
import watershed_workflow
import watershed_workflow.ui
import logging
watershed_workflow.ui.setup_logging(1,None)

import numpy as np
import rasterio
import fiona
import watershed_workflow.daymet

import os,sys
import numpy as np
import pandas
from matplotlib import pyplot as plt
from matplotlib import cm as pcm
import logging
import pandas as pd
import geopandas as gpd
# import seaborn as sns
import shapely
import copy
import scipy
import datetime
import netCDF4 as nc

import watershed_workflow 
import watershed_workflow.source_list
import watershed_workflow.ui
import watershed_workflow.utils
import watershed_workflow.plot
import watershed_workflow.mesh
import watershed_workflow.condition
# import watershed_workflow.densify_rivers_hucs
# import watershed_workflow.create_river_mesh
watershed_workflow.ui.setup_logging(1,None)

# import pygeoutils as geoutils
# from pynhd import NLDI
import ipympl


In [3]:
sources = watershed_workflow.source_list.get_default_sources()
name = 'Neches' # Neches + Alligator Bayou domain
hucs = ['1202'] # a list of HUCs to run
huc_level = None # if provided, an int setting the level at which to include HUC boundaries
crs = watershed_workflow.crs.daymet_crs()
crs

<Derived Projected CRS: +proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=- ...>
Name: unknown
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- undefined
Coordinate Operation:
- name: unknown
- method: Lambert Conic Conformal (2SP)
Datum: Unknown based on WGS84 ellipsoid
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [4]:
# Get shapefile for the domain of analysis. Use the USGS gauge ID to pull data
# sources = watershed_workflow.source_list.get_default_sources()
# my_hucs = []
# for huc in hucs:
#     _, ws = watershed_workflow.get_hucs(sources['HUC'], huc, huc_level, crs)
#     my_hucs.extend(ws)

# Option 1: Get the basin gauge boundary directly from the shp
# Note: if pynhd has an "verification certificates" error, change ssl to False for the ar.retrieve_text
# and ar.retrieve_json in the nwis.py file 
# Users/gpq/opt/anaconda/envs/Env_test/lib/python3.11/site-packages/pygeohydro/nwis.py , line 146, 537

# Most general option, still a hack, is to change the async_retriever.py function line 337 to : ssl: SSLContext | bool | None = False,
# Add verify=False to the requests.get to all the .py files in watershed_workflow/sources/*.py

In [4]:
# Get watershed boundary. This was defined from mesh_Neches_nhd.ipynb

path_watershed = '../data-processed/basin/complete_watershed_domain.shp'

watershed_neches = gpd.read_file(path_watershed)
_, watershed_neches=watershed_workflow.get_shapes(path_watershed,in_crs = watershed_neches.crs,out_crs = watershed_workflow.crs.daymet_crs())


# # save watershed_containing_basin as a shapefile, this is in shapely format
# gdf_basin = gpd.GeoDataFrame(geometry=[watershed_containing_basin], crs=crs) 
# gdf_basin.to_file('../data-processed/basin/complete_watershed_domain.shp')

2024-04-24 15:02:45,305 - fiona.ogrext - INFO: Failed to auto identify EPSG: 7
2024-04-24 15:02:45,334 - root - INFO: 
2024-04-24 15:02:45,335 - root - INFO: Loading shapes
2024-04-24 15:02:45,335 - root - INFO: ------------------------------
2024-04-24 15:02:45,335 - root - INFO: Loading file: '../data-processed/basin/complete_watershed_domain.shp'
2024-04-24 15:02:45,389 - fiona.ogrext - INFO: Failed to auto identify EPSG: 7
2024-04-24 15:02:45,391 - root - INFO: ... found 1 shapes
2024-04-24 15:02:45,392 - root - INFO: Converting to shapely


In [5]:
# Get watershed in the right format
watershed = watershed_workflow.split_hucs.SplitHUCs(watershed_neches)

In [7]:

# # gauge_ID='08041780' # National Water Service streamflow gauge ID (USGS ID)
# # watershed_gauge = NLDI().get_basins([gauge_ID])
# # watershed_gauge.to_file(f'../data-processed/basin/watershed_gaugeID_{gauge_ID}.shp')
# # path_shp_basin = 'data/basin/watershed_gauge' # Path to save shp
# HUC4 = '1202' # HUC4 ID
# gpd_watershed = gpd.read_file(f'../data-processed/basin/watershed_HUC4_{HUC4}.shp')
# _, watershed_gauge=watershed_workflow.get_shapes(f'../data-processed/basin/watershed_HUC4_{HUC4}.shp',in_crs = gpd_watershed.crs,out_crs = watershed_workflow.crs.daymet_crs())

# watershed_area = watershed_gauge[0].area/1e6 # Area in [km2]
# # my_hucs = watershed_gauge[0]
# my_hucs = [my_hucs[0].intersection(watershed_gauge[0])]
# print('Drainage area = ', watershed_area, 'km^2')

# # Get watershed in the right format
# watershed = watershed_workflow.split_hucs.SplitHUCs(my_hucs)

## import watershed

In [6]:
# crs, watershed = watershed_workflow.get_split_form_shapes(watershed_shapefile)
# logging.info(f'crs: {crs}')

bounds = watershed.exterior().bounds
bounds


(371518.6107982158,
 -1389737.0764046875,
 583664.8140665749,
 -1050151.8582880897)

## Download Daymet

returned raw data has `dim(nband, ncol, nrow)`

Note that here we need two files -- the actual data and the typical year data.

The first cell downloads the raw data and generates the actual data file used by ATS, the second cell averages days, smooths the data, and writes a typical year.

In [9]:
# Note: if the data is not being downloded then probably the id had changed and the line [60] on manager_daymet.py must be updated. 
# line [60] --> URL = "https://thredds.daac.ornl.gov/thredds/ncss/ornldaac/2129/daymet_v4_daily_na_{variable}_{year}.nc"

# bounds = watershed.exterior().bounds

# # a dictionary of outputs -- will include all filenames generated
# 2016-2018
# start = "1-2016"
# end = "365-2018"
# outputs = {}
# outputs['daymet_filename'] = f'../../../global_data/rainfall_input_ATS/{name}/{name}_DayMet_2016_2018.h5'

# dat, x, y = watershed_workflow.daymet.collectDaymet(bounds, crs, start, end)

# # Save data into HD5 format
# ats = watershed_workflow.daymet.daymetToATS(dat)
# attrs = watershed_workflow.daymet.getAttrs(bounds, start, end)

# watershed_workflow.daymet.writeHDF5(ats, x, y, attrs, outputs['daymet_filename'])

In [10]:

# # a dictionary of outputs -- will include all filenames generated
# 2010-2020

# bounds = watershed.exterior().bounds

# start = "1-2010"
# end = "365-2020"
# outputs = {}
# outputs['daymet_filename'] = f'../../../global_data/rainfall_input_ATS/{name}/{name}_DayMet_2010_2020.h5'

# dat, x, y = watershed_workflow.daymet.collectDaymet(bounds, crs, start, end)

# # Save data into HD5 format
# ats = watershed_workflow.daymet.daymetToATS(dat)
# attrs = watershed_workflow.daymet.getAttrs(bounds, start, end)
# watershed_workflow.daymet.writeHDF5(ats, x, y, attrs, outputs['daymet_filename'])



In [11]:
# For 20 years of transient simulations

In [12]:
# Hack to download and convert all the data for large domains
bounds = watershed.exterior().bounds

# 2000-2020
# start = "1-1980"
start_year = 2000
end_year = 2020
start = f"1-{start_year}"
end = f"365-{end_year}"
outputs = {}
outputs['daymet_filename'] = f'../../../global_data/rainfall_input_ATS/{name}/{name}_DayMet_{start_year}_{end_year}.h5'

attrs = watershed_workflow.daymet.getAttrs(bounds, start, end)

# Initialize the `ats` dictionary outside the loop
ats = {}

for year in range(start_year, end_year + 1):
    start = f"1-{year}"
    end = f"365-{year}"
    # Collect DayMet data for the given year
    dat, x, y = watershed_workflow.daymet.collectDaymet(bounds, crs, start, end)
    # Convert DayMet data to ATS format for the year
    year_ats = watershed_workflow.daymet.daymetToATS(dat)  
    del dat
    # If `ats` is empty, initialize it with the data from the first year
    if not ats:
        ats = year_ats
    else:
        # For each key in the yearly data, append the new data to the existing data in `ats`
        for key in year_ats.keys():
            # Concatenate the new array with the existing one for the same key
            ats[key] = np.concatenate((ats[key], year_ats[key]))
# The atribute "time [s]" is set to zero at each year, so we need to set it mannualy. with increaments of 1 day (24*60*60)
ats['time [s]'] = np.arange(0, len(ats['time [s]'])*24*60*60, 24*60*60)


# Save data into HD5 format
watershed_workflow.daymet.writeHDF5(ats, x, y, attrs, outputs['daymet_filename'])



2024-04-02 18:51:48,813 - root - INFO: downloading variables: ['tmin', 'tmax', 'prcp', 'srad', 'vp', 'swe', 'dayl']
2024-04-02 18:51:48,821 - root - INFO: Collecting DayMet file to tile bounds: [-96.10390000000001, 29.2867, -93.5867, 32.5624]
2024-04-02 18:51:48,822 - root - INFO:   Using existing: /Users/gpq/ORNL/Projects/ATS_runs/global_data/meteorology/daymet/daymet_tmin_2000_32.5624x-96.1039_29.2867x-93.5867.nc
2024-04-02 18:51:49,060 - root - INFO: Collecting DayMet file to tile bounds: [-96.10390000000001, 29.2867, -93.5867, 32.5624]
2024-04-02 18:51:49,061 - root - INFO:   Using existing: /Users/gpq/ORNL/Projects/ATS_runs/global_data/meteorology/daymet/daymet_tmax_2000_32.5624x-96.1039_29.2867x-93.5867.nc
2024-04-02 18:51:49,288 - root - INFO: Collecting DayMet file to tile bounds: [-96.10390000000001, 29.2867, -93.5867, 32.5624]
2024-04-02 18:51:49,289 - root - INFO:   Using existing: /Users/gpq/ORNL/Projects/ATS_runs/global_data/meteorology/daymet/daymet_prcp_2000_32.5624x-96.

In [ ]:
# For Spin-up. We generate a cyclic series representing 40 years of data.
# This is created using the last 10 years of the data. In theory, we could use 40 years of data to create the spin-up data, but this wouldn't capture potential trends
# Besides, using 40 years of data would require a lot of memory.

In [7]:
# Hack to download and convert all the data for large domains 
bounds = watershed.exterior().bounds

# 2000-2020
# start = "1-1980"
start_year = 2010 
end_year = 2020
start = f"1-{start_year}"
end = f"365-{end_year}"
outputs = {}
outputs['daymet_spinup_filename'] = f'../../../global_data/rainfall_input_ATS/{name}/{name}_DayMet_typical_2000_2020.h5' #Note the name is just to  
# outputs['daymet_filename'] = f'../../../global_data/rainfall_input_ATS/{name}/{name}_DayMet_test.h5'
attrs = watershed_workflow.daymet.getAttrs(bounds, start, end)
dat, x, y = watershed_workflow.daymet.collectDaymet(bounds, crs, start, end)
# Convert DayMet data to ATS format for the year
ats_typ = watershed_workflow.daymet.daymetToATS(dat, smooth=True, smooth_filter=True, nyears=1)  
# clear data from memory
del dat

# calculate the basin-averaged, annual-averaged precip rate
precip_total = ats_typ['precipitation rain [m s^-1]'] + ats_typ['precipitation snow [m SWE s^-1]']
mean_precip = precip_total.mean()
print(f'Mean annual precip rate [m s^-1] = {mean_precip}')

# clear data from memory
del precip_total
# Create a cyclic series representing 20 years of data
nyears = 20
for key in ats_typ.keys():
    ats_typ[key] = np.concatenate([ats_typ[key]]*nyears)

# The atribute "time [s]" is set to zero at each year, so we need to set it mannualy. with increaments of 1 day (24*60*60)
ats_typ['time [s]'] = np.arange(0, len(ats_typ['time [s]'])*24*60*60, 24*60*60)

# Save data into HD5 format
watershed_workflow.daymet.writeHDF5(ats_typ, x, y, attrs, outputs['daymet_spinup_filename'])

2024-04-24 15:03:20,228 - root - INFO: downloading variables: ['tmin', 'tmax', 'prcp', 'srad', 'vp', 'swe', 'dayl']
2024-04-24 15:03:20,239 - root - INFO: Collecting DayMet file to tile bounds: [-96.10390000000001, 29.2867, -93.5867, 32.5624]
2024-04-24 15:03:20,241 - root - INFO:   Using existing: /Users/gpq/ORNL/Projects/ATS_runs/global_data/meteorology/daymet/daymet_tmin_2010_32.5624x-96.1039_29.2867x-93.5867.nc
2024-04-24 15:03:20,522 - root - INFO: Collecting DayMet file to tile bounds: [-96.10390000000001, 29.2867, -93.5867, 32.5624]
2024-04-24 15:03:20,523 - root - INFO:   Using existing: /Users/gpq/ORNL/Projects/ATS_runs/global_data/meteorology/daymet/daymet_tmax_2010_32.5624x-96.1039_29.2867x-93.5867.nc
2024-04-24 15:03:20,793 - root - INFO: Collecting DayMet file to tile bounds: [-96.10390000000001, 29.2867, -93.5867, 32.5624]
2024-04-24 15:03:20,794 - root - INFO:   Using existing: /Users/gpq/ORNL/Projects/ATS_runs/global_data/meteorology/daymet/daymet_prcp_2010_32.5624x-96.

Mean annual precip rate [m s^-1] = 2.7757994897942437e-08


2024-04-24 15:04:36,036 - root - INFO: Writing HDF5 file: ../../../global_data/rainfall_input_ATS/Neches/Neches_DayMet_typical_2000_2020.h5


In [8]:
# Create a shorter spin-up for testing the model (only 1 year)

# Hack to download and convert all the data for large domains 
bounds = watershed.exterior().bounds

# 2000-2020
# start = "1-1980"
start_year = 2010 
end_year = 2020
start = f"1-{start_year}"
end = f"365-{end_year}"
outputs = {}
outputs['daymet_spinup_filename'] = f'../../../global_data/rainfall_input_ATS/{name}/{name}_DayMet_typical_2020.h5' #Note the name is just to  
# outputs['daymet_filename'] = f'../../../global_data/rainfall_input_ATS/{name}/{name}_DayMet_test.h5'
attrs = watershed_workflow.daymet.getAttrs(bounds, start, end)
dat, x, y = watershed_workflow.daymet.collectDaymet(bounds, crs, start, end)
# Convert DayMet data to ATS format for the year
ats_typ = watershed_workflow.daymet.daymetToATS(dat, smooth=True, smooth_filter=True, nyears=1)  
# clear data from memory
del dat

# calculate the basin-averaged, annual-averaged precip rate
precip_total = ats_typ['precipitation rain [m s^-1]'] + ats_typ['precipitation snow [m SWE s^-1]']
mean_precip = precip_total.mean()
print(f'Mean annual precip rate [m s^-1] = {mean_precip}')

# clear data from memory
del precip_total
# Create a cyclic series representing 1 year of data
nyears = 1
for key in ats_typ.keys():
    ats_typ[key] = np.concatenate([ats_typ[key]]*nyears)

# The atribute "time [s]" is set to zero at each year, so we need to set it mannualy. with increaments of 1 day (24*60*60)
ats_typ['time [s]'] = np.arange(0, len(ats_typ['time [s]'])*24*60*60, 24*60*60)

# Save data into HD5 format
watershed_workflow.daymet.writeHDF5(ats_typ, x, y, attrs, outputs['daymet_spinup_filename'])

2024-04-22 13:26:23,811 - root - INFO: downloading variables: ['tmin', 'tmax', 'prcp', 'srad', 'vp', 'swe', 'dayl']
2024-04-22 13:26:23,838 - root - INFO: Collecting DayMet file to tile bounds: [-96.10390000000001, 29.2867, -93.5867, 32.5624]
2024-04-22 13:26:23,839 - root - INFO:   Using existing: /Users/gpq/ORNL/Projects/ATS_runs/global_data/meteorology/daymet/daymet_tmin_2010_32.5624x-96.1039_29.2867x-93.5867.nc
2024-04-22 13:26:24,130 - root - INFO: Collecting DayMet file to tile bounds: [-96.10390000000001, 29.2867, -93.5867, 32.5624]
2024-04-22 13:26:24,130 - root - INFO:   Using existing: /Users/gpq/ORNL/Projects/ATS_runs/global_data/meteorology/daymet/daymet_tmax_2010_32.5624x-96.1039_29.2867x-93.5867.nc
2024-04-22 13:26:24,412 - root - INFO: Collecting DayMet file to tile bounds: [-96.10390000000001, 29.2867, -93.5867, 32.5624]
2024-04-22 13:26:24,413 - root - INFO:   Using existing: /Users/gpq/ORNL/Projects/ATS_runs/global_data/meteorology/daymet/daymet_prcp_2010_32.5624x-96.

Mean annual precip rate [m s^-1] = 2.7757994897942437e-08


In [8]:
# Convert DayMet data to the new format, version ATS 1.4


# Read dir with function
ats_src_dir = os.getenv('ATS_SRC_DIR')
# Construct the full path to the converter script
path_convert_daymet = os.path.join(ats_src_dir, 'tools', 'utils', 'rh_to_vp.py')


In [ ]:


# Construct the command to run the conversion script
command = ['python', path_convert_daymet, '--inplace', '../../../global_data/rainfall_input_ATS/Neches/Neches_DayMet_typical_2020.h5']
# Run the command
result = subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# Output the results of the subprocess
print("Output:", result.stdout.decode())
print("Error:", result.stderr.decode())

In [ ]:
# Construct the command to run the conversion script
command = ['python', path_convert_daymet, '--inplace', '../../../global_data/rainfall_input_ATS/Neches/Neches_DayMet_typical_2020.h5']
# Run the command
result = subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# Output the results of the subprocess
print("Output:", result.stdout.decode())
print("Error:", result.stderr.decode())

In [9]:
# Construct the command to run the conversion script
command = ['python', path_convert_daymet, '--inplace', '../../../global_data/rainfall_input_ATS/Neches/Neches_DayMet_typical_2000_2020.h5']
# Run the command
result = subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# Output the results of the subprocess
print("Output:", result.stdout.decode())
print("Error:", result.stderr.decode())

Output: 
Error: 


In [15]:
# Construct the command to run the conversion script
command = ['python', path_convert_daymet, '--inplace', '../../../global_data/rainfall_input_ATS/Neches/Neches_DayMet_2019_2020.h5']
# Run the command
result = subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# Output the results of the subprocess
print("Output:", result.stdout.decode())
print("Error:", result.stderr.decode())

Output: 
Error: 


In [16]:
# Construct the command to run the conversion script
command = ['python', path_convert_daymet, '--inplace', '../../../global_data/rainfall_input_ATS/Neches/Neches_DayMet_2000_2020.h5']
# Run the command
result = subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# Output the results of the subprocess
print("Output:", result.stdout.decode())
print("Error:", result.stderr.decode())

Output: 
Error: 
